In [ ]:
import pandas as pd
import os

In [ ]:
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.','-')

%env USER_NAME={USER_NAME}

In [ ]:
!gsutil -u $GOOGLE_PROJECT -m ls -lah gs://fc-aou-datasets-controlled/v7/microarray/plink_v7.1/

#  Plink array QC for subgroups

In [ ]:
cur_dir = '/home/jupyter/workspaces/genomicsofinfectiousdiseasesusceptibility/meta'

## Make parameter file to run parallel jobs

In [ ]:
# Initialize an Empty DataFrame
param = []

my_bucket = os.getenv('WORKSPACE_BUCKET')

for anc in ["eur", "afr", "eas", "amr", "oth", "mid", "sas"]:
    param.append({'--input keep_file' : f"{my_bucket}/data/meta/plink_array/sumstats_{anc}/plink_array_{anc}.keep",
                  '--input remove_file' : f"{my_bucket}/data/meta/plink_array/remove_het_outliers/plink_array_het_{anc}.remove",
                  '--output-recursive outpath' : f"{my_bucket}/data/meta/plink_array/qc_{anc}",
                  '--env ancestry' : anc})

param = pd.DataFrame(param)
print(param)

In [ ]:
PARAMETER_FILENAME = f'{cur_dir}/plinkarray_qc_bygroupPARAM.tsv'

param.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

%env PARAMETER_FILENAME={PARAMETER_FILENAME}

## Script to compute sumstats

Since this is a dsub job, we need the input files to be copied to the container and the input files out of the container. So they are specified within the dsub jobs

In [ ]:
%%writefile script.sh

#!/bin/bash
    
# This script takes in the plink array on AOU (non imputed)
# and performs QC. Keep makes sure that only the relevant
# ancestry is kept. Remove removes the heterozygosity outliers.
# For sex chromosome, we only keep pseudoautosomal regions.

set -o errexit
set -o nounset

plink1.9 --bfile "${plinkarray_path}/arrays" \
    --keep "${keep_file}" \
    --remove "${remove_file}" \
    --chr 1-22 XY \
    --geno 0.05 \
    --maf 0.05 \
    --hwe 10e-12 \
    --mind 0.1 \
    --indep-pairwise 50 10 0.1 \
    --exclude "${high_LD}" \
    --memory 9000 \
    --make-bed \
    --out "${outpath}/plinkarray_qc_${ancestry}"

In [ ]:
!gsutil mv script.sh ${WORKSPACE_BUCKET}/data/scripts/bash/plink_array/plinkarray_qc_bygroup.sh

In [ ]:
!gsutil cat ${WORKSPACE_BUCKET}/data/scripts/bash/plink_array/plinkarray_qc_bygroup.sh

## Submit dsub job

This script makes a nice folder structure within GCP and does the job over there. The next cell outputs a python env variable called job_id

In [ ]:
#!gsutil rm -r ${WORKSPACE_BUCKET}/data/logs/plinkarray-qc-bygroup

In [ ]:
%%bash --out job_id

source ~/aou_dsub.bash

aou_dsub \
    --image biocontainers/plink1.9:v1.90b6.6-181012-1-deb_cv1 \
    --min-ram 10 \
    --min-cores 1 \
    --input-recursive plinkarray_path="gs://fc-aou-datasets-controlled/v7/microarray/plink_v7.1" \
    --input high_LD="gs://fc-secure-1792472b-ed73-4937-b557-928a360bc420/data/raw/high_LD_hg38_GRCh38.txt" \
    --logging "${WORKSPACE_BUCKET}/data/logs/{job-name}/{job-id}-{task-id}.log" \
    --tasks "${PARAMETER_FILENAME}" \
    --script "${WORKSPACE_BUCKET}/data/scripts/bash/plink_array/plinkarray_qc_bygroup.sh"

In [ ]:
%env JOB_ID={job_id}

In [ ]:
%%bash

dstat \
    --provider google-cls-v2 \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --users "${USER_NAME}" \
    --jobs "${JOB_ID}" \
    --status '*'

## Read log files

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/qc_eur/plinkarray_qc_eur.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/qc_afr/plinkarray_qc_afr.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/qc_amr/plinkarray_qc_amr.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/qc_eas/plinkarray_qc_eas.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/qc_mid/plinkarray_qc_mid.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/qc_sas/plinkarray_qc_sas.log"

In [ ]:
%%bash

gsutil cat "${WORKSPACE_BUCKET}/data/meta/plink_array/qc_oth/plinkarray_qc_oth.log"